In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings

from scipy.cluster.hierarchy import linkage, dendrogram

from sklearn.preprocessing import PolynomialFeatures, StandardScaler, LabelEncoder, MinMaxScaler, RobustScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, SGDClassifier
from sklearn.svm import SVR, SVC
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics import (mean_squared_error, mean_absolute_error, root_mean_squared_error, 
    r2_score, accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, silhouette_score, davies_bouldin_score)
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances, euclidean_distances

from imblearn.under_sampling import TomekLinks
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

from statsmodels.stats.stattools import jarque_bera, omni_normtest, durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

import statsmodels.api as sm

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import re
import nltk
import codecs

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

import chardet

from gensim.models import Word2Vec

import multiprocessing

import os
import sys

sys.path.append(os.getcwd() + '/numi_libs/')

import file_io as fio
import data_eda as eda
import data_forge as frg
import data_model as dmd

nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')

warnings.filterwarnings('ignore')

seed = 420
np.random.seed(seed)

In [ ]:
filename_c = "patient-data.csv"
filename_n = 'curse-of-dimensionality.xlsx'

df_c = fio.readCsvToDF(filename_c)

eda.initialAssessment(df_c, "Patient Dataset")


file_path = 'curse-of-dimensionality.xlsx'
df_n = pd.read_excel(filename_n, sheet_name='Sheet3')

eda.initialAssessment(df_n, "Curse of Dimensionality Dataset")
print("assessment complete")

In [ ]:
eda.profileFeatures(df_n)

In [ ]:
x_tr, x_te, y_tr, y_te = frg.ttSplit(df_n, "y")
x_tr_trf, x_te_trf, _= frg.yjTransform(x_tr, x_te)
x_tr_trf, x_te_trf, _= frg.minMaxScale(x_tr_trf, x_te_trf)
eda.profileFeatures(x_tr_trf)

In [ ]:
# eda.generateEDATables(df)
# eda.profileFeatures(df)
# eda.profileIntercorrelation(df)
# eda.boxPlotData(df)
# df = pd.concat([T1.reset_index(drop=True),T2.reset_index(drop=True)], axis=1)

algorithms = {
    'Linear Regression': LinearRegression(),
    #'SVM Regression': SVR(kernel='linear'),  # Adjust kernel as needed
    #'RandomForest': RandomForestRegressor(),
    #'XGBoost': GradientBoostingRegressor(),
    #'knn': KNeighborsRegressor(),
    #'Neural Network-10-5-5': MLPRegressor(hidden_layer_sizes=[10, 5, 5], max_iter=20000),
}

pca = [0]

dmd.applyAndReportRegressors(x_tr, x_te, y_tr, y_te, algorithms, pca, False, "Vanilla")
dmd.applyAndReportRegressors(x_tr_trf, x_te_trf, y_tr, y_te, algorithms, pca, False, "Transformed")

#eda.profileFeatures(x_data_tfsc)

In [ ]:
vifs = eda.probeVifRemoval(df_n.drop(columns = ['y']))

print(vifs)

In [ ]:
x_tr_trf_vif = x_tr_trf.drop(columns = ['x1', 'x8', 'x5', 'x3', 'x7'])
x_te_trf_vif = x_te_trf.drop(columns = ['x1', 'x8', 'x5', 'x3', 'x7'])
dmd.applyAndReportRegressors(x_tr_trf_vif, x_te_trf_vif, y_tr, y_te, algorithms, pca, True, "Transformed + VIF")


In [ ]:
eda.performPcaAnalysis(x_tr_trf)

In [ ]:
eda.performPcaAnalysis(x_tr)

In [ ]:
pca = [0, 2]
dmd.applyAndReportRegressors(x_tr_trf, x_te_trf, y_tr, y_te, 
                             algorithms, pca, False, "Transformed")